# CSIRO Biomass: Small DINO Notebook

This notebook sets up a compact DINO-based regressor for the CSIRO biomass competition using a small backbone when available.

In [ ]:
# If needed, install dependencies in this environment:
# !pip install torch torchvision timm pandas numpy scikit-learn pillow tqdm

In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
TARGET_ORDER = [
    "Dry_Clover_g",
    "Dry_Dead_g",
    "Dry_Green_g",
    "Dry_Total_g",
    "GDM_g",
]


@dataclass
class Config:
    dataset_dir: Path = Path("csiro-biomass")
    model_name: str | None = None
    image_size: int = 224
    batch_size: int = 8
    epochs: int = 6
    freeze_epochs: int = 1
    learning_rate: float = 2e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    seed: int = 42
    output_model: Path = Path("dino_small_model_best.pt")
    output_submission: Path = Path("dino_small_submission.csv")


cfg = Config()
cfg

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_small_dino_model() -> str:
    available = set(timm.list_models("*dino*"))
    preferred = [
        "vit_small_patch14_dinov2.lvd142m",
        "vit_small_patch14_reg4_dinov2.lvd142m",
        "vit_small_patch16_224.dino",
        "vit_small_patch8_224.dino",
    ]

    for name in preferred:
        if name in available:
            return name

    fallback = sorted(
        [
            name
            for name in available
            if "small" in name and ("dinov2" in name or ".dino" in name)
        ]
    )
    if fallback:
        return fallback[0]

    raise ValueError(
        "No small DINO model found in this timm install. "
        "Run timm.list_models('*dino*') to inspect what is available."
    )


if cfg.model_name is None:
    cfg.model_name = pick_small_dino_model()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(cfg.seed)

print(f"Using device: {device}")
print(f"Using model: {cfg.model_name}")

In [ ]:
class BiomassImageDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        dataset_dir: Path,
        image_col: str,
        target_cols: list[str] | None,
        transform,
    ) -> None:
        self.dataframe = dataframe.reset_index(drop=True)
        self.dataset_dir = dataset_dir
        self.image_col = image_col
        self.target_cols = target_cols
        self.transform = transform

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        image_path = self.dataset_dir / row[self.image_col]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.target_cols is None:
            return image

        target = row[self.target_cols].values.astype(np.float32)
        return image, torch.from_numpy(target)


def prepare_train_dataframe(train_csv: Path) -> pd.DataFrame:
    train_df = pd.read_csv(train_csv)
    pivoted = train_df.pivot_table(
        index=[
            "image_path",
            "Sampling_Date",
            "State",
            "Species",
            "Pre_GSHH_NDVI",
            "Height_Ave_cm",
        ],
        columns="target_name",
        values="target",
    ).reset_index()

    missing = [target for target in TARGET_ORDER if target not in pivoted.columns]
    if missing:
        raise ValueError(f"Missing targets in pivoted data: {missing}")

    return pivoted

In [ ]:
def build_transforms(model_name: str, image_size: int):
    data_cfg = timm.data.resolve_model_data_config(
        timm.create_model(model_name, pretrained=False)
    )

    eval_transform = timm.data.create_transform(
        input_size=(3, image_size, image_size),
        mean=data_cfg["mean"],
        std=data_cfg["std"],
        interpolation=data_cfg["interpolation"],
        is_training=False,
    )

    train_transform = timm.data.create_transform(
        input_size=(3, image_size, image_size),
        mean=data_cfg["mean"],
        std=data_cfg["std"],
        interpolation=data_cfg["interpolation"],
        is_training=True,
        hflip=0.5,
        vflip=0.2,
        color_jitter=0.2,
    )
    return train_transform, eval_transform


def create_dataloaders(cfg: Config, train_pivoted: pd.DataFrame):
    state_counts = train_pivoted["State"].value_counts()
    can_stratify = len(state_counts) > 1 and state_counts.min() >= 2
    stratify_col = train_pivoted["State"] if can_stratify else None

    train_data, val_data = train_test_split(
        train_pivoted,
        test_size=0.2,
        random_state=cfg.seed,
        stratify=stratify_col,
    )

    train_transform, eval_transform = build_transforms(cfg.model_name, cfg.image_size)

    train_dataset = BiomassImageDataset(
        dataframe=train_data,
        dataset_dir=cfg.dataset_dir,
        image_col="image_path",
        target_cols=TARGET_ORDER,
        transform=train_transform,
    )
    val_dataset = BiomassImageDataset(
        dataframe=val_data,
        dataset_dir=cfg.dataset_dir,
        image_col="image_path",
        target_cols=TARGET_ORDER,
        transform=eval_transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    return train_loader, val_loader, eval_transform, train_data, val_data

In [ ]:
class DinoRegressor(nn.Module):
    def __init__(self, model_name: str, num_targets: int = 5, dropout: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        feature_dim = self.backbone.num_features
        self.regressor = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_targets),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.regressor(features)

    def set_backbone_trainable(self, trainable: bool) -> None:
        for parameter in self.backbone.parameters():
            parameter.requires_grad = trainable

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    device: torch.device,
) -> float:
    model.train()
    running_loss = 0.0

    for images, targets in tqdm(dataloader, desc="Train", leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


def validate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()
    running_loss = 0.0
    mae_sum = 0.0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Val", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, targets)
            mae = torch.mean(torch.abs(outputs - targets))

            running_loss += loss.item() * images.size(0)
            mae_sum += mae.item() * images.size(0)

    val_loss = running_loss / len(dataloader.dataset)
    val_mae = mae_sum / len(dataloader.dataset)
    return val_loss, val_mae


def train_model(cfg: Config, train_loader: DataLoader, val_loader: DataLoader):
    model = DinoRegressor(model_name=cfg.model_name, num_targets=len(TARGET_ORDER)).to(device)
    criterion = nn.SmoothL1Loss(beta=2.0)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_val_loss = float("inf")

    for epoch in range(1, cfg.epochs + 1):
        model.set_backbone_trainable(epoch > cfg.freeze_epochs)
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_mae = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:02d}/{cfg.epochs} "
            f"| train_loss={train_loss:.4f} "
            f"| val_loss={val_loss:.4f} "
            f"| val_mae={val_mae:.4f} "
            f"| lr={lr:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": cfg.model_name,
                    "targets": TARGET_ORDER,
                    "image_size": cfg.image_size,
                },
                cfg.output_model,
            )
            print(f"Saved best checkpoint to {cfg.output_model}")

    checkpoint = torch.load(cfg.output_model, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

In [ ]:
train_pivoted = prepare_train_dataframe(cfg.dataset_dir / "train.csv")
train_loader, val_loader, eval_transform, train_data, val_data = create_dataloaders(cfg, train_pivoted)

print(f"Train rows: {len(train_data)}")
print(f"Val rows:   {len(val_data)}")
print(train_pivoted[TARGET_ORDER].describe().T[["mean", "std", "min", "max"]])

In [ ]:
# Train the model (adjust cfg.epochs / cfg.batch_size above for faster or stronger runs).
model = train_model(cfg, train_loader, val_loader)

In [ ]:
def predict_unique_images(
    model: DinoRegressor,
    image_paths,
    dataset_dir: Path,
    transform,
    batch_size: int,
    num_workers: int,
) -> pd.DataFrame:
    unique_df = pd.DataFrame({"image_path": list(dict.fromkeys(image_paths))})
    test_dataset = BiomassImageDataset(
        dataframe=unique_df,
        dataset_dir=dataset_dir,
        image_col="image_path",
        target_cols=None,
        transform=transform,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    preds = []
    model.eval()
    with torch.no_grad():
        for images in tqdm(test_loader, desc="Predict", leave=False):
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            preds.append(outputs.cpu().numpy())

    pred_values = np.vstack(preds)
    pred_values = np.maximum(pred_values, 0.0)
    pred_df = pd.DataFrame(pred_values, columns=TARGET_ORDER)
    pred_df["image_path"] = unique_df["image_path"].values
    return pred_df


def create_submission(cfg: Config, model: DinoRegressor, eval_transform) -> pd.DataFrame:
    test_df = pd.read_csv(cfg.dataset_dir / "test.csv")
    pred_df = predict_unique_images(
        model=model,
        image_paths=test_df["image_path"].values,
        dataset_dir=cfg.dataset_dir,
        transform=eval_transform,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
    )

    pred_lookup = pred_df.set_index("image_path")
    rows = []
    for _, row in test_df.iterrows():
        target = float(pred_lookup.loc[row["image_path"], row["target_name"]])
        rows.append({"sample_id": row["sample_id"], "target": target})

    submission = pd.DataFrame(rows)
    submission.to_csv(cfg.output_submission, index=False)
    print(f"Saved submission to {cfg.output_submission} | shape={submission.shape}")
    return submission

In [ ]:
# Generate a Kaggle submission file.
submission = create_submission(cfg, model, eval_transform)
submission.head()

In [ ]:
from pathlib import Path

import timm
import torch
import torch.nn as nn


class DinoRegressor(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_targets: int = 5,
        hidden_dim: int | None = None,
        dropout: float = 0.3,
        use_pretrained_backbone: bool = False,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=use_pretrained_backbone,
            num_classes=0,
            global_pool="avg",
        )
        feature_dim = self.backbone.num_features

        if hidden_dim is None:
            self.regressor = nn.Sequential(
                nn.LayerNorm(feature_dim),
                nn.Dropout(dropout),
                nn.Linear(feature_dim, num_targets),
            )
        else:
            self.regressor = nn.Sequential(
                nn.LayerNorm(feature_dim),
                nn.Linear(feature_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, num_targets),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.regressor(features)


checkpoint_path = Path("dino_small_model_best.pt")
if not checkpoint_path.exists():
    checkpoint_path = Path("dino_model") / "dino_small_model_best.pt"

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint["model_state_dict"]
model_name = checkpoint.get("model_name", "vit_small_patch14_dinov2")
num_targets = len(checkpoint.get("targets", ["t1", "t2", "t3", "t4", "t5"]))
image_size = int(checkpoint.get("image_size", 224))

hidden_dim = None
if "regressor.4.weight" in state_dict:
    hidden_dim = int(state_dict["regressor.1.weight"].shape[0])

model = DinoRegressor(
    model_name=model_name,
    num_targets=num_targets,
    hidden_dim=hidden_dim,
    dropout=0.3,
    use_pretrained_backbone=False,
).to(device)

missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
model.eval()


def _shape_str(output):
    if isinstance(output, (list, tuple)) and output:
        output = output[0]
    if not torch.is_tensor(output):
        return "multiple"
    shape = list(output.shape)
    if shape:
        shape[0] = None
    return str(tuple(shape))


def print_collapsed_dino_summary(net: nn.Module, input_size=(3, 224, 224), batch_size=2):
    hooks = []
    shape_map = {}

    def make_hook(layer_name: str):
        def hook_fn(_module, _inputs, outputs):
            shape_map[layer_name] = _shape_str(outputs)

        return hook_fn

    # Keep DINO hidden as one row.
    hooks.append(net.backbone.register_forward_hook(make_hook("dino_layer")))

    # Show regressor internals.
    for idx, module in enumerate(net.regressor):
        layer_name = f"regressor.{idx}:{module.__class__.__name__}"
        hooks.append(module.register_forward_hook(make_hook(layer_name)))

    prev_mode = net.training
    net.eval()
    with torch.no_grad():
        dummy = torch.randn(batch_size, *input_size, device=device)
        _ = net(dummy)

    for h in hooks:
        h.remove()
    net.train(prev_mode)

    line = "=" * 110
    print(line)
    print(f"{'Layer (type)':<46} {'Output Shape':<32} {'Param #':>12}")
    print(line)

    # Row 1: collapsed DINO backbone.
    dino_params = sum(p.numel() for p in net.backbone.parameters())
    print(f"{'dino_layer':<46} {shape_map.get('dino_layer', 'n/a'):<32} {dino_params:>12,}")

    # Rows 2+: expanded regressor layers.
    for idx, module in enumerate(net.regressor):
        layer_name = f"regressor.{idx}:{module.__class__.__name__}"
        layer_params = sum(p.numel() for p in module.parameters())
        print(f"{layer_name:<46} {shape_map.get(layer_name, 'n/a'):<32} {layer_params:>12,}")

    print(line)

    total_params = sum(p.numel() for p in net.parameters())
    trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    non_trainable_params = total_params - trainable_params

    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    print(f"Non-trainable params: {non_trainable_params:,}")


print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Backbone: {model_name}")
print(f"Head type: {'deep (2 linear layers)' if hidden_dim is not None else 'shallow (1 linear layer)'}")
if hidden_dim is not None:
    print(f"Inferred hidden_dim: {hidden_dim}")

if missing_keys:
    print(f"Missing keys: {len(missing_keys)}")
if unexpected_keys:
    print(f"Unexpected keys: {len(unexpected_keys)}")

print("\nKeras-like model summary (DINO collapsed, regressor expanded)")
print_collapsed_dino_summary(model, input_size=(3, image_size, image_size), batch_size=2)